# EDA: BiciBog — Sistema de Bicicletas Compartidas de Bogotá

Análisis exploratorio de datos para apoyar decisiones de expansión de estaciones, mantenimiento de bicicletas y estrategias de movilidad sostenible.

**Dataset:** `bicicletas_compartidas_bogota.csv`


## 0. Carga y exploración inicial de los datos

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import unicodedata
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

df = pd.read_csv('bicicletas_compartidas_bogota.csv')
print("Dimensiones:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
# Valores faltantes por columna
df.isna().sum()


In [ ]:
df.describe(include='all').T


### 0.1 Limpieza de categorías inconsistentes

Se detectan inconsistencias de mayúsculas/tildes en `tipo_bicicleta` (`Eléctrica` vs `electrica`) y en `clima` (`Lluvia ligera` vs `lluvia ligera`). Se normalizan antes de continuar.

In [ ]:
def normalizar(s):
    if pd.isna(s):
        return s
    return unicodedata.normalize('NFKD', s.strip().lower()).encode('ascii', 'ignore').decode()

print("Antes de normalizar:")
print(df['tipo_bicicleta'].value_counts(), "\n")
print(df['clima'].value_counts())

df['tipo_bicicleta'] = df['tipo_bicicleta'].apply(normalizar).str.capitalize()
df['clima'] = df['clima'].apply(normalizar)

print("\nDespués de normalizar:")
print(df['tipo_bicicleta'].value_counts(), "\n")
print(df['clima'].value_counts())


## 1. Relación entre `distancia_km` y `duracion_min`

### 1.a Coeficiente de correlación de Pearson

In [ ]:
sub = df.dropna(subset=['distancia_km', 'duracion_min'])
print("N usado para la correlación:", len(sub))

r, p_valor = stats.pearsonr(sub['distancia_km'], sub['duracion_min'])
print(f"Coeficiente de Pearson r = {r:.4f}")
print(f"p-value = {p_valor:.2e}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(sub['distancia_km'], sub['duracion_min'], alpha=0.3, s=15)
plt.xlabel('Distancia (km)')
plt.ylabel('Duración (min)')
plt.title(f'Distancia vs. Duración (r = {r:.2f})')
plt.tight_layout()
plt.show()


**Interpretación (b):** correlación positiva y fuerte (r ≈ 0.72). A mayor distancia, mayor duración; la distancia explica ~52% (r²) de la variabilidad de la duración.

**¿Correlación implica causalidad? (c):** No necesariamente. Aunque existe un mecanismo físico plausible (recorrer más distancia toma más tiempo), el coeficiente de Pearson solo mide asociación lineal, no causalidad. Podrían existir factores de confusión (tráfico, tipo de bicicleta, ruta).

### 1.d ¿Una correlación alta implica que TODOS los viajes largos en distancia tengan duración alta?

Veamos ejemplos concretos que contradicen esa idea:

In [ ]:
# Ejemplo: distancia relativamente alta pero duración corta
alta_dist_baja_dur = sub[(sub['distancia_km'] >= sub['distancia_km'].quantile(0.85))].sort_values('duracion_min').head(5)
print("Viajes de distancia alta pero duración corta:")
print(alta_dist_baja_dur[['viaje_id', 'distancia_km', 'duracion_min', 'tipo_bicicleta']])

# Ejemplo: distancia baja pero duración larga
baja_dist_alta_dur = sub[(sub['distancia_km'] <= sub['distancia_km'].quantile(0.15))].sort_values('duracion_min', ascending=False).head(5)
print("\nViajes de distancia baja pero duración larga:")
print(baja_dist_alta_dur[['viaje_id', 'distancia_km', 'duracion_min', 'tipo_bicicleta']])


**Conclusión:** no, una correlación alta describe una tendencia general, no una regla universal. Existen viajes individuales (como los de arriba) que se apartan bastante de la tendencia promedio.

## 2. Detección de outliers con la regla del rango intercuartílico (IQR)

In [ ]:
def resumen_iqr(serie, nombre):
    s = serie.dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    outliers = s[(s < limite_inf) | (s > limite_sup)]
    print(f"--- {nombre} ---")
    print(f"Q1 = {Q1:.3f}")
    print(f"Q3 = {Q3:.3f}")
    print(f"IQR = {IQR:.3f}")
    print(f"Límite inferior = {limite_inf:.3f}")
    print(f"Límite superior = {limite_sup:.3f}")
    print(f"N° de outliers = {len(outliers)} ({100 * len(outliers) / len(s):.2f}%)\n")
    return limite_inf, limite_sup

low_d, high_d = resumen_iqr(df['distancia_km'], 'distancia_km')
low_t, high_t = resumen_iqr(df['duracion_min'], 'duracion_min')


In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.boxplot(df['distancia_km'].dropna())
plt.title('Boxplot: distancia_km')
plt.subplot(1, 2, 2)
plt.boxplot(df['duracion_min'].dropna())
plt.title('Boxplot: duracion_min')
plt.tight_layout()
plt.show()


### Los 5 registros más extremos

Se combinan ambas variables (suma de valores z absolutos) para encontrar los casos más atípicos en conjunto, y se calcula la velocidad implícita como chequeo de plausibilidad física.

In [ ]:
sub = df.dropna(subset=['distancia_km', 'duracion_min']).copy()
sub['z_dist'] = (sub['distancia_km'] - sub['distancia_km'].mean()) / sub['distancia_km'].std()
sub['z_dur'] = (sub['duracion_min'] - sub['duracion_min'].mean()) / sub['duracion_min'].std()
sub['extremeness'] = sub['z_dist'].abs() + sub['z_dur'].abs()
sub['velocidad_kmh'] = sub['distancia_km'] / (sub['duracion_min'] / 60)

top5 = sub.sort_values('extremeness', ascending=False).head(5)
top5[['viaje_id', 'distancia_km', 'duracion_min', 'velocidad_kmh', 'tipo_bicicleta', 'clima', 'precipitacion_mm']]


**Tabla de clasificación (completar/discutir en el informe):**

| Registro | Clasificación | Justificación |
|---|---|---|
| V000070 (180.0 min, 3.71 km) | Probablemente error | Duración "tope" de 180.0 min repetida en otro registro; sugiere timeout del sistema |
| V002333 (180.0 min, 3.62 km) | Probablemente error | Mismo patrón exacto de 180.0 min |
| V000844 (8.9 min, 31.2 km) | Probablemente error | Velocidad implícita ≈ 210 km/h, físicamente imposible |
| V002209 (22.3 min, 27.5 km) | Probablemente error | Velocidad implícita ≈ 74 km/h, imposible incluso en bici eléctrica, y bajo lluvia |
| V001789 (125.0 min, 4.65 km) | Ambiguo | Duración larga para distancia corta; podría ser un viaje real con paradas, o bici no devuelta a tiempo |

**¿Deberían eliminarse todos?** No automáticamente: los que implican velocidades físicamente imposibles sí deberían excluirse o corregirse; el caso "ambiguo" requiere más investigación antes de descartarlo.

## 3. Correlación excluyendo los outliers

In [ ]:
mask_outlier = (
    (sub['distancia_km'] < low_d) | (sub['distancia_km'] > high_d) |
    (sub['duracion_min'] < low_t) | (sub['duracion_min'] > high_t)
)
print("Total de outliers (unión de ambas variables):", mask_outlier.sum(),
      f"({100 * mask_outlier.sum() / len(sub):.2f}%)")

limpio = sub[~mask_outlier]

r_original, _ = stats.pearsonr(sub['distancia_km'], sub['duracion_min'])
r_limpio, _ = stats.pearsonr(limpio['distancia_km'], limpio['duracion_min'])

print(f"\nr original       (n={len(sub)})  = {r_original:.4f}")
print(f"r sin outliers    (n={len(limpio)}) = {r_limpio:.4f}")
print(f"Diferencia = {r_limpio - r_original:+.4f}")


**Interpretación:**
- **(a)** Sí cambió considerablemente: subió de 0.72 a 0.89 (en vez de bajar), porque los outliers detectados son justamente los que rompen la relación lineal (velocidades imposibles, duraciones "topadas").
- **(b)** Esto indica que la relación real entre distancia y duración es más fuerte y consistente de lo que sugería el análisis con datos crudos; el r original estaba "contaminado" por errores de captura/censura.
- **(c)** No es correcto reportar solo una: lo adecuado es mostrar ambas y explicar la diferencia, documentando el criterio usado para excluir los outliers.

## 4. Tabla por tipo de bicicleta

In [ ]:
tabla_bici = df.groupby('tipo_bicicleta').agg(
    n_viajes=('viaje_id', 'count'),
    distancia_prom_km=('distancia_km', 'mean'),
    duracion_prom_min=('duracion_min', 'mean'),
    duracion_mediana_min=('duracion_min', 'median')
).round(2)
tabla_bici


**¿Las eléctricas se asocian con viajes más cortos en tiempo?** Sí: con una distancia promedio prácticamente igual a las mecánicas, las eléctricas registran una duración promedio y mediana notablemente menor, consistente con la asistencia eléctrica permitiendo mayor velocidad.

## 5. ¿El clima determina la duración del viaje?

In [ ]:
print("=== Por clima (todos los viajes) ===")
tabla_clima_todos = df.groupby('clima').agg(
    n_viajes=('viaje_id', 'count'),
    distancia_prom_km=('distancia_km', 'mean'),
    duracion_prom_min=('duracion_min', 'mean'),
    duracion_mediana_min=('duracion_min', 'median')
).round(2)
print(tabla_clima_todos)


In [ ]:
print("=== Por clima (solo viajes entre 2 y 4 km) ===")
filtrado = df[(df['distancia_km'] >= 2) & (df['distancia_km'] <= 4)]
tabla_clima_filtrado = filtrado.groupby('clima').agg(
    n_viajes=('viaje_id', 'count'),
    distancia_prom_km=('distancia_km', 'mean'),
    duracion_prom_min=('duracion_min', 'mean'),
    duracion_mediana_min=('duracion_min', 'median')
).round(2)
print(tabla_clima_filtrado)


In [ ]:
tabla_clima_todos['duracion_prom_min'].plot(kind='bar', figsize=(6,4), title='Duración promedio por clima (todos los viajes)')
plt.ylabel('Duración promedio (min)')
plt.tight_layout()
plt.show()

tabla_clima_filtrado['duracion_prom_min'].plot(kind='bar', figsize=(6,4), title='Duración promedio por clima (viajes 2-4 km)', color='orange')
plt.ylabel('Duración promedio (min)')
plt.tight_layout()
plt.show()


**Evaluación crítica de "el clima determina la duración del viaje":**

- Sin controlar por distancia, la duración promedio varía muy poco entre climas (~0.5 min de rango).
- Al controlar por distancia (2–4 km), las diferencias se reducen aún más y dejan de mostrar un patrón claro.
- Comparar todos los viajes sin controlar la distancia puede ser engañoso porque `distancia_km` actúa como **variable de confusión**: es la que realmente explica buena parte de la variación en duración (r ≈ 0.72–0.89), no el clima.
- **Conclusión:** no se puede afirmar que el clima *cause* cambios en la duración; a lo sumo hay una asociación muy débil que se diluye al controlar por distancia.

## 6. Conclusión para el equipo (máx. 200 palabras)

> La variable con mayor poder explicativo sobre la duración de los viajes es la **distancia recorrida** (r ≈ 0.72 en bruto, r ≈ 0.89 tras depurar outliers evidentes), mientras que el clima muestra, a lo sumo, una asociación marginal que se diluye al controlar por distancia — probablemente actuando como confusor más que como causa. Entre los casos atípicos destaca el par de viajes con duración exacta de 180.0 minutos: la coincidencia sugiere un tope técnico del sistema (bicicleta no devuelta a tiempo) más que un tiempo real de uso, y merece revisión con el operador antes de tratarse como dato válido. Sería tentador concluir que "el clima no afecta en nada el comportamiento de los usuarios", pero los datos no permiten descartarlo del todo: solo muestran que su efecto sobre la *duración*, controlando distancia, es pequeño; no se evaluó su efecto sobre la *demanda* de viajes. Una pregunta pendiente es si el clima influye más en cuántos viajes se hacen que en cuánto duran. Se recomienda: (1) normalizar categorías de texto inconsistentes, (2) validar/excluir velocidades implícitas imposibles, y (3) confirmar con el operador si existe un tope de 180 minutos en el sistema.